In [1]:
import os
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as data
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchsummary import summary
import matplotlib.pyplot as plt
from PIL import Image
import time

# **Download Data**

In [2]:
ROOT = './data'

train_data = datasets.MNIST(
    ROOT,
    train=True,
    download=True
)

test_data = datasets.MNIST(
    ROOT,
    train=False,
    download=True
)

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 9.91M/9.91M [00:00<00:00, 16.0MB/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 28.9k/28.9k [00:00<00:00, 505kB/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 1.65M/1.65M [00:00<00:00, 4.42MB/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 4.54k/4.54k [00:00<00:00, 5.05MB/s]

Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw



# **Preproccessing**

In [3]:
# Split train:valid = 0.9:0.1
VALID_RATIO = 0.1
n_train_ex = int(len(train_data) * (1 - VALID_RATIO))
n_val_ex = len(train_data) - n_train_ex

train_data_split, valid_data = data.random_split(train_data, lengths=[n_train_ex, n_val_ex])

# Normalization
data_all = train_data_split.dataset.data.float() / 255
mean = data_all.mean()
std = data_all.std()

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean.item()], std=[std.item()])
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean.item()], std=[std.item()])
])

train_data_split.dataset.transform = train_transform
valid_data.dataset.transform = train_transform


# Create DataLoader
BATCH_SIZE = 256
from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    train_data_split,
    shuffle=True,
    batch_size=BATCH_SIZE
)

valid_dataloader = DataLoader(
    valid_data,
    shuffle=False,
    batch_size=BATCH_SIZE
)



# **Model Construction**

In [15]:
class LeNetClassifier(nn.Module):
  def __init__(self, num_classes):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, padding=2) # Changed padding to a valid value
    self.avgpool1 = nn.AvgPool2d(kernel_size=2)
    self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, padding=0) # Explicitly set padding to 0 to make sure there is no confusion
    self.avgpool2 = nn.AvgPool2d(kernel_size=2)
    self.flatten = nn.Flatten()
    self.fc_1 = nn.Linear(16 * 5 * 5, 120)  # Adjust input size to match the output of convolutional layers
    self.fc_2 = nn.Linear(120, 84)
    self.fc_3 = nn.Linear(84, num_classes)

  def forward(self, inputs):
    outputs = self.conv1(inputs)
    outputs = self.avgpool1(outputs) # Changed from ouptuts to outputs to correctly apply pooling
    outputs = F.relu(outputs)
    outputs = self.conv2(outputs)
    outputs = self.avgpool2(outputs)
    outputs = F.relu(outputs)
    outputs = self.flatten(outputs)
    outputs = self.fc_1(outputs)
    outputs = self.fc_2(outputs)
    outputs = self.fc_3(outputs)
    return  outputs

# **Training**

In [22]:
def train(model, optimizer, criterion, train_dataloader, device, epoch=0, log_interval=50):
  model.train()
  total_acc, total_count = 0, 0
  losses = []
  start_time = time.time()

  for idx, (inputs, labels) in enumerate(train_dataloader):
      inputs = inputs.to(device)
      labels = labels. to(device)

      optimizer.zero_grad()
      predictions = model(inputs)

      # compute loss
      loss = criterion(predictions, labels)
      losses.append(loss.item())

      # backward
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
      optimizer.step()
      total_acc += (predictions.argmax(1) == labels).sum().item()
      total_count +=  labels.size(0)
      if idx % log_interval == 0 and idx > 0:
          elapsed = time.time() - start_time
          print ("| epoch {:3d} | {:5d}/{:5d} batches ""| accuracy {:8.3f}". format (epoch , idx , len ( train_dataloader ), total_acc / total_count))
          total_acc, total_count = 0, 0
          start_time = time.time()

  epoch_acc = total_acc / total_count
  epoch_loss = sum(losses) / len(losses)
  return epoch_acc, epoch_loss

def evaluate(model, criterion, valid_dataloader, device):
  model.eval()
  total_acc, total_count = 0, 0
  losses = []

  with torch.no_grad():
    for idx, (inputs, labels) in enumerate(valid_dataloader):
      inputs = inputs.to(device)
      labels = labels.to(device)

      predictions = model(inputs)

      loss = criterion(predictions, labels)
      losses.append(loss.item())

      total_acc += (predictions.argmax(1) == labels).sum().item()
      total_count += labels.size(0)
  epoch_acc = total_acc / total_count
  epoch_loss = sum(losses) / len(losses)
  return epoch_acc, epoch_loss


In [24]:
num_classes = len(train_data_split.dataset.classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

lenet_model = LeNetClassifier(num_classes)
lenet_model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(lenet_model.parameters())\

num_epochs = 10
save_model = '/content/model'

train_accs, train_losses = [], []
eval_accs, eval_losses = [], []
best_loss_eval = 100

for epoch in range(1, num_epochs + 1):
  epoch_start_time = time.time()
  # Training
  train_acc, train_loss = train(lenet_model, optimizer, criterion, train_dataloader, device, epoch)
  train_accs.append(train_acc)
  train_losses.append(train_loss)

  # Evaluation
  eval_acc, eval_loss = evaluate(lenet_model, criterion, valid_dataloader, device)
  eval_accs.append(eval_acc)
  train_losses.append(eval_loss)

  # Save best model
  if eval_loss < best_loss_eval:
    torch.save(lenet_model.state_dict(), save_model + "/lenet_model.pt")

  # Print loss and accuracy at the end of each epoch
  print("-" * 59)
  print(
      "| End of epoch {:3d} | Time: {:5.2f}s | Train Accuracy: {:8.3f} | Train Loss: {:8.3f} |"
      " Valid Accuracy: {:8.3f} | Valid Loss: {:8.3f} |".format(
          epoch,
          time.time() - epoch_start_time,
          train_acc,
          train_loss,
          eval_acc,
          eval_loss
      )
  )
  print("-" * 59)

  # Load the best model
  lenet_model.load_state_dict(torch.load(save_model + '/lenet_model.pt'))
  lenet_model.eval()



| epoch   1 |    50/  211 batches | accuracy    0.648
| epoch   1 |   100/  211 batches | accuracy    0.868
| epoch   1 |   150/  211 batches | accuracy    0.905
| epoch   1 |   200/  211 batches | accuracy    0.928
-----------------------------------------------------------
| End of epoch   1 | Time: 13.40s | Train Accuracy:    0.953 | Train Loss:    0.526 | Valid Accuracy:    0.939 | Valid Loss:    0.209 |
-----------------------------------------------------------


<ipython-input-24-36603fe886ac>:49: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  lenet_model.load_state_dict(torch.load(save_model + '/lenet_model.pt'))


| epoch   2 |    50/  211 batches | accuracy    0.944
| epoch   2 |   100/  211 batches | accuracy    0.957
| epoch   2 |   150/  211 batches | accuracy    0.960
| epoch   2 |   200/  211 batches | accuracy    0.965
-----------------------------------------------------------
| End of epoch   2 | Time: 13.32s | Train Accuracy:    0.965 | Train Loss:    0.144 | Valid Accuracy:    0.959 | Valid Loss:    0.136 |
-----------------------------------------------------------
| epoch   3 |    50/  211 batches | accuracy    0.966
| epoch   3 |   100/  211 batches | accuracy    0.971
| epoch   3 |   150/  211 batches | accuracy    0.972
| epoch   3 |   200/  211 batches | accuracy    0.974
-----------------------------------------------------------
| End of epoch   3 | Time: 13.33s | Train Accuracy:    0.974 | Train Loss:    0.097 | Valid Accuracy:    0.971 | Valid Loss:    0.102 |
-----------------------------------------------------------
| epoch   4 |    50/  211 batches | accuracy    0.975
| 